# Week 7 – Roll A & B & C & D (CSV-põhine)

Andmed loetakse ainult lokaalsetest CSV failidest (`sales.csv`, `customers.csv`).
Lõpptulemus on puhastatud `df`

## Roll A – Andmete laadimine

Laadin `sales` ja `customers` CSV failidest pandas DataFrame'i, kontrolli struktuuri ja liida tabelid `customer_id` põhjal.


In [18]:
import pandas as pd

# Loeme andmed lokaalsetest CSV failidest (encoding='utf-8-sig' eemaldab BOM-i).
df_sales = pd.read_csv("sales.csv", encoding="utf-8-sig")
df_customers = pd.read_csv("customers.csv", encoding="utf-8-sig")

print("Sales shape:", df_sales.shape)
display(df_sales.head())

print("Customers shape:", df_customers.shape)
display(df_customers.head())

print("\nSales dtypes:")
print(df_sales.dtypes)

Sales shape: (15234, 11)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


Customers shape: (3150, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2001,Eha,Aas,eha.aas@telia.ee,+372 8713 1455,Tallinn,2024-02-27,NaN,1973
1,2002,Aivar,Kõiv,aivar.koiv@outlook.com,+372 8943 8684,Haapsalu,2025-01-09,bronze,1988
2,2003,Maris,Rebane,maris.rebane@telia.ee,+372 5918 5726,Tartu,2021-02-03,NaN,1999
3,2004,Jaak,Talvik,jaak.talvik@mail.ee,+372 8554 4232,Tallinn,2023-11-12,silver,1974
4,2005,Raivo,Koppel,raivo.koppel@yahoo.com,+372 5298 4365,Tallinn,2023-05-22,bronze,2004



Sales dtypes:
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object


In [19]:
# Liidan tabelid customer_id põhjal (left join säilitab kõik müügiread).
df = pd.merge(df_sales, df_customers, on="customer_id", how="left")

print("Liidetud (merged) shape:", df.shape)
print("\nLiidetud dtypes:")
print(df.dtypes)
display(df.head())

# Kontroll: kas võtmeveerud on olemas?
vajalikud = ["customer_id", "sale_date", "total_price", "email"]
print("\nOlemasolevad võtmeveerud:", [c for c in vajalikud if c in df.columns])

Liidetud (merged) shape: (15234, 19)

Liidetud dtypes:
sale_id                int64
invoice_id               str
sale_date                str
customer_id          float64
product_id             int64
quantity               int64
unit_price           float64
total_price          float64
channel                  str
store_location           str
payment_method           str
first_name               str
last_name                str
email                    str
phone                    str
city                     str
registration_date        str
loyalty_tier             str
birth_year           float64
dtype: object


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0



Olemasolevad võtmeveerud: ['customer_id', 'sale_date', 'total_price', 'email']


## Roll B – Andmete puhastamine

**Light clean:** põhipuhastus on duplikaatide eemaldamine primäarvõtme `sale_id` järgi (Supabase'is on `sale_id` primäarvõti) ja kuupäevade parsimine.

NULL `customer_id` ja negatiivsed `total_price` **raporteeritakse, kuid säilitatakse** – NULL customer_id tähendab külalist/sidumata klienti ja negatiivsed on tagastused. Nende eemaldamine muudaks tulemuse Supabase allikast erinevaks.

In [20]:
# 1) Roll A väljund (df). Esialgne shape.
print("Esialgne shape:", df.shape)

# 2) Duplikaadid: eemalda primäarvõtme sale_id järgi 
print("\nTäisrea duplikaadid:", df.duplicated().sum())
print("Korduvad sale_id:", df["sale_id"].duplicated().sum())
df = df.drop_duplicates(subset="sale_id")
print("Shape pärast sale_id dedup:", df.shape)

# 3) NULL väärtused 
print("\nNULL-id:")
print(df.isnull().sum())

# 4) Kuupäevade parsimine (mixed + dayfirst -> nii YYYY-MM-DD kui DD/MM/YYYY).
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce", format="mixed", dayfirst=True)
print("\nsale_date tüüp:", df["sale_date"].dtype, "| NaT väärtusi:", df["sale_date"].isna().sum())

# 5) Outlier'id: negatiivsed total_price (tagastused) – raporteeri, kuid ei eemalda.
print("Negatiivseid total_price:", (df["total_price"] < 0).sum())

# 6) Puhastusraport.
print("\n===== PUHASTUSRAPORT =====")
print("Lõplik shape:", df.shape)
print("Unikaalseid kliente:", df["customer_id"].nunique())
print("Kuupäevavahemik:", df["sale_date"].min().date(), "kuni", df["sale_date"].max().date())

Esialgne shape: (15234, 19)

Täisrea duplikaadid: 4086
Korduvad sale_id: 5116
Shape pärast sale_id dedup: (10118, 19)

NULL-id:
sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64

sale_date tüüp: datetime64[us] | NaT väärtusi: 0
Negatiivseid total_price: 195

===== PUHASTUSRAPORT =====
Lõplik shape: (10118, 19)
Unikaalseid kliente: 2551
Kuupäevavahemik: 2023-01-01 kuni 2026-12-03


In [21]:
# ==========================================
# RFM ANALÜÜS
# ==========================================

import pandas as pd
import numpy as np


# 1. Viitekuupäev
today = pd.to_datetime('2025-02-28')

# Andmete laadimine (kohanda failiteed vastavalt)
df = pd.read_csv('sales_clean.csv')
# Kui sale_date on stringina, teisenda kuupäevaks
df['sale_date'] = pd.to_datetime(df['sale_date'])

# ------------------------------------------
# 2. Recency arvutamine
# ------------------------------------------
last_purchase = (
    df.groupby('customer_id')['sale_date']
      .max()
      .reset_index()
)

last_purchase['recency_days'] = (
    today - pd.to_datetime(last_purchase['sale_date'])
).dt.days

recency = last_purchase[['customer_id', 'recency_days']]

# ------------------------------------------
# 3. Frequency arvutamine
# ------------------------------------------
frequency = (
    df.groupby('customer_id')['sale_id']
      .count()
      .reset_index()
      .rename(columns={'sale_id': 'frequency'})
)

# ------------------------------------------
# 4. Monetary arvutamine
# ------------------------------------------
monetary = (
    df.groupby('customer_id')['total_price']
      .sum()
      .reset_index()
      .rename(columns={'total_price': 'monetary'})
)

# ------------------------------------------
# 5. RFM tabeli loomine
# ------------------------------------------
rfm = recency.merge(frequency, on='customer_id')
rfm = rfm.merge(monetary, on='customer_id')

# ------------------------------------------
# RFM skoorid (1-5)
# ------------------------------------------

# Recency - vastupidine skoor
rfm['R_score'] = pd.qcut(
    rfm['recency_days'].rank(method='first'),
    5,
    labels=[5, 4, 3, 2, 1]
)

# Frequency
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

# Monetary
rfm['M_score'] = pd.qcut(
    rfm['monetary'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

# Teisendame täisarvudeks
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)

# ------------------------------------------
# 6. RFM koguskoor
# ------------------------------------------
rfm['RFM_Score'] = (
    rfm['R_score'] +
    rfm['F_score'] +
    rfm['M_score']
)

# ------------------------------------------
# Segmentide määramine
# ------------------------------------------
def assign_segment(score):
    if score >= 13:
        return 'VIP Champions'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'Potential'
    elif score >= 4:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

# ------------------------------------------
# RFM tulemuste eelvaade
# ------------------------------------------
print("RFM tabel:")
display(rfm.head())

# ------------------------------------------
# Segmentide kokkuvõte
# ------------------------------------------
segment_summary = (
    rfm.groupby('Segment')
       .agg(Customer_Count=('customer_id', 'count'))
       .reset_index()
)

segment_summary['Percentage'] = (
    segment_summary['Customer_Count']
    / segment_summary['Customer_Count'].sum()
    * 100
).round(2)

segment_summary = segment_summary.sort_values(
    by='Customer_Count',
    ascending=False
)

print("\nSegmentide kokkuvõte:")
display(segment_summary)

# ------------------------------------------
# Kvaliteedikontroll
# ------------------------------------------
print("\nR_score jaotus:")
print(rfm['R_score'].value_counts().sort_index())

print("\nF_score jaotus:")
print(rfm['F_score'].value_counts().sort_index())

print("\nM_score jaotus:")
print(rfm['M_score'].value_counts().sort_index())

print("\nPuuduvate segmentide arv:")
print(rfm['Segment'].isna().sum())

RFM tabel:


,customer_id,recency_days,frequency,monetary,R_score,F_score,M_score,RFM_Score,Segment
0,2001.0,91,2,203.92,4,1,1,6,At Risk
1,2004.0,71,2,1198.56,4,1,4,9,Potential
2,2005.0,164,4,959.60,3,4,4,11,Loyal
3,2006.0,536,1,327.06,1,1,1,3,Lost
4,2007.0,29,1,318.63,5,1,1,7,Potential



Segmentide kokkuvõte:


,Segment,Customer_Count,Percentage
3,Potential,756,29.64
2,Loyal,701,27.48
0,At Risk,523,20.50
4,VIP Champions,450,17.64
1,Lost,121,4.74



R_score jaotus:
R_score
1    510
2    510
3    510
4    510
5    511
Name: count, dtype: int64

F_score jaotus:
F_score
1    511
2    510
3    510
4    510
5    510
Name: count, dtype: int64

M_score jaotus:
M_score
1    511
2    510
3    510
4    510
5    510
Name: count, dtype: int64

Puuduvate segmentide arv:
0


In [22]:
# ==========================================
# RFM VISUALISEERIMINE JA ÄRITÕLGENDUS
# ==========================================

import plotly.express as px
import pandas as pd

# Kui sinu RFM tabelis on veerg monetary, siis teeme sellest vihjes kasutatud nime
if 'monetary_value' not in rfm.columns:
    rfm['monetary_value'] = rfm['monetary']

# ------------------------------------------
# 1. Segmentide jaotus
# ------------------------------------------

segment_counts = (
    rfm.groupby('Segment')
       .agg(Customer_Count=('customer_id', 'count'))
       .reset_index()
       .sort_values('Customer_Count', ascending=False)
)

fig1 = px.bar(
    segment_counts,
    x='Segment',
    y='Customer_Count',
    text='Customer_Count',
    title='RFM segmentide jaotus',
    labels={
        'Segment': 'Kliendisegment',
        'Customer_Count': 'Klientide arv'
    }
)

fig1.update_traces(textposition='outside')
fig1.update_layout(showlegend=False)

fig1.show()

# ------------------------------------------
# 2. Hajuvusdiagramm: Recency vs Monetary
# ------------------------------------------

fig2 = px.scatter(
    rfm,
    x='recency_days',
    y='monetary_value',
    color='Segment',
    size='frequency',
    hover_data=['customer_id', 'RFM_Score'],
    title='UrbanStyle kliendisegmendid RFM analüüsi põhjal',
    labels={
        'recency_days': 'Päevi viimasest ostust',
        'monetary_value': 'Kogukulutus (€)',
        'frequency': 'Ostude arv',
        'Segment': 'Segment'
    }
)

fig2.show()

# ------------------------------------------
# 3. Top 10 VIP klienti kogukulutuse järgi
# ------------------------------------------

top_vip = (
    rfm[rfm['Segment'] == 'VIP Champions']
    .nlargest(10, 'monetary_value')
)

fig3 = px.bar(
    top_vip,
    x='customer_id',
    y='monetary_value',
    text='monetary_value',
    title='Top 10 VIP klienti kogukulutuse järgi',
    labels={
        'customer_id': 'Kliendi ID',
        'monetary_value': 'Kogukulutus (€)'
    }
)

fig3.update_traces(texttemplate='%{text:.2f} €', textposition='outside')
fig3.update_layout(showlegend=False)

fig3.show()

# ------------------------------------------
# 4. Äritõlgenduse arvutused
# ------------------------------------------

total_customers = rfm['customer_id'].nunique()
total_revenue = rfm['monetary_value'].sum()

vip_customers = rfm[rfm['Segment'] == 'VIP Champions']
vip_count = vip_customers['customer_id'].nunique()
vip_revenue = vip_customers['monetary_value'].sum()
vip_revenue_share = vip_revenue / total_revenue * 100

at_risk_customers = rfm[rfm['Segment'] == 'At Risk']
at_risk_count = at_risk_customers['customer_id'].nunique()

lost_customers = rfm[rfm['Segment'] == 'Lost']
lost_count = lost_customers['customer_id'].nunique()

# ------------------------------------------
# 5. Äritõlgendus Markole
# ------------------------------------------

print("ÄRITÕLGENDUS MARKOLE")
print("-" * 40)

print(
    f"RFM analüüsi põhjal on UrbanStyle'il {vip_count} VIP Champions klienti, "
    f"kes moodustavad {vip_revenue_share:.1f}% kogu kliendikäibest. "
    f"Need kliendid on kõige väärtuslikumad, sest nad ostavad hiljuti, sageli ja suure summa eest. "
    f"Samas on At Risk segmendis {at_risk_count} klienti ning Lost segmendis {lost_count} klienti, "
    f"mis viitab võimalikule kliendikao riskile. "
    f"Fookus peaks olema VIP klientide hoidmisel ja At Risk klientide tagasivõitmisel."
)

print("\nSOOVITUSED")
print("-" * 40)

print("1. Loo VIP programm: eripakkumised, varajane ligipääs kampaaniatele ja personaalsed soodustused VIP Champions klientidele.")
print("2. Käivita win-back kampaania At Risk klientidele: saada neile personaalne allahindlus või meeldetuletus viimati ostetud kategooria põhjal.")
print("3. Loo nurture programm Potential klientidele: suuna nad kordusostule e-kirjade, kampaaniate ja lojaalsuspakkumistega.")

ÄRITÕLGENDUS MARKOLE
----------------------------------------
RFM analüüsi põhjal on UrbanStyle'il 450 VIP Champions klienti, kes moodustavad 43.0% kogu kliendikäibest. Need kliendid on kõige väärtuslikumad, sest nad ostavad hiljuti, sageli ja suure summa eest. Samas on At Risk segmendis 523 klienti ning Lost segmendis 121 klienti, mis viitab võimalikule kliendikao riskile. Fookus peaks olema VIP klientide hoidmisel ja At Risk klientide tagasivõitmisel.

SOOVITUSED
----------------------------------------
1. Loo VIP programm: eripakkumised, varajane ligipääs kampaaniatele ja personaalsed soodustused VIP Champions klientidele.
2. Käivita win-back kampaania At Risk klientidele: saada neile personaalne allahindlus või meeldetuletus viimati ostetud kategooria põhjal.
3. Loo nurture programm Potential klientidele: suuna nad kordusostule e-kirjade, kampaaniate ja lojaalsuspakkumistega.
